# Turbomachinery Degradation & RUL Prognostics with BiLSTM + Attention

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohammedbasra-star/turbomachinery-degradation-lstm-/blob/main/notebooks/rul_quickstart_demo.ipynb)
[![License: MIT](https://img.shields.io/badge/License-MIT-blue.svg)](https://opensource.org/licenses/MIT)
[![Python 3.10+](https://img.shields.io/badge/python-3.10+-brightgreen.svg)](https://www.python.org/downloads/)
[![PyTorch](https://img.shields.io/badge/PyTorch-2.0+-ee4c2c.svg)](https://pytorch.org/)

## 1. Turbomachinery Prognostic Background & Physics
In gas turbines and aircraft turbofans, rotating components (e.g. high-pressure compressor blades, bearings, combustor liners) experience continuous mechanical, aerodynamic, and thermal degradation. 

The **NASA C-MAPSS (Commercial Modular Aero-Propulsion System Simulation)** benchmark dataset models the degradation of 100 run-to-failure engines. 

### Asymmetric Industrial Safety Metric (NASA Scoring Function)
In aviation and turbomachinery maintenance, late predictions ($y_{pred} > y_{true}$) risk catastrophic in-flight failure and are penalized much more heavily than early predictions ($y_{pred} < y_{true}$):
$$S = \sum_{i=1}^{N} s_i, \quad s_i = \begin{cases} e^{-\frac{d_i}{13}} - 1 & \text{if } d_i < 0 \text{ (Early Maintenance / Conservative)} \\ e^{\frac{d_i}{10}} - 1 & \text{if } d_i \ge 0 \text{ (Late Prediction / Critical Safety Hazard)} \end{cases}$$
where $d_i = y_{pred} - y_{true}$.

In [ ]:
# Install dependencies if running on Google Colab
import sys
try:
    import torch, pandas, numpy, matplotlib, seaborn
except ImportError:
    !pip install -q torch pandas numpy matplotlib seaborn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch Device: {device}")

## 2. Ingesting & Visualizing Sensor Degradation Trajectories

In [ ]:
from data.dataset import load_cmapss_data, prepare_sliding_windows, INFORMATIVE_SENSORS
train_df, test_df, true_rul = load_cmapss_data()

print(f"Train Engines: {train_df['unit_nr'].nunique()} | Test Engines: {test_df['unit_nr'].nunique()}")
train_df.head()

In [ ]:
plt.figure(figsize=(9, 4.5))
for u in [1, 2, 5, 8]:
    sub = train_df[train_df["unit_nr"] == u]
    plt.plot(sub["time_cycles"], sub["s_3"], label=f"Engine #{u}", linewidth=1.8)

plt.xlabel("Operational Flight Cycles", fontweight="bold")
plt.ylabel("HPC Outlet Temperature T30 (°R)", fontweight="bold")
plt.title("Thermodynamic Degradation: HPC Temperature Drift Over Engine Lifespan", fontweight="bold")
plt.legend()
plt.grid(True, linestyle=":", alpha=0.6)
plt.show()

## 3. Sliding Window Preprocessing

In [ ]:
data_dict = prepare_sliding_windows(train_df, test_df, true_rul, sequence_length=30, max_rul=125.0)
X_train, y_train = data_dict["X_train"], data_dict["y_train"]
X_test, y_test = data_dict["X_test"], data_dict["y_test"]
print(f"Sequenced Train Windows: {X_train.shape} | Test Fleet Windows: {X_test.shape}")

## 4. BiLSTM with Temporal Self-Attention Mechanism Definition

In [ ]:
from models.lstm_attention import BiLSTMAttention
model = BiLSTMAttention(in_features=14, hidden_dim=64, num_layers=2).to(device)
print(model)

## 5. Test Fleet Verification & Attention Heatmap Inspection

In [ ]:
import os
ckpt_path = "../results/best_model.pt"
if not os.path.exists(ckpt_path):
    ckpt_path = "../checkpoints/best_model.pt"

if os.path.exists(ckpt_path):
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()
    with torch.no_grad():
        preds, attns = model(torch.tensor(X_test).to(device), return_attention=True)
        preds = np.maximum(0.0, preds.cpu().numpy().ravel())
        attns = attns.cpu().numpy()
        rmse = np.sqrt(np.mean((preds - y_test)**2))
        print(f"Evaluated BiLSTM+Attention -> Fleet Test RMSE: {rmse:.2f} Flight Cycles")
        
        # Plot attention heatmap for first 5 test engines
        plt.figure(figsize=(9, 4))
        plt.imshow(attns[:5], cmap="YlOrRd", aspect="auto")
        plt.colorbar(label="Attention Weight α_t")
        plt.title("Temporal Attention Distribution (Cycles Influencing Failure Prediction)", fontweight="bold")
        plt.xlabel("Time Steps in Window (1 to 30)")
        plt.ylabel("Test Engine Unit")
        plt.show()
else:
    print("Train checkpoint will be evaluated when training finishes.")